# 2 · Arquitectura del agente: qué decide el LLM y qué hace el código

1. **El LLM (qwen2.5:7b)** recibe la pregunta y los esquemas de 5 herramientas; en un bucle de hasta 3 turnos elige herramientas, argumentos y cuándo abstenerse.
2. **El código** valida tipos y rangos con Pydantic, comprueba que cada argumento esté *anclado* en la pregunta (país, ciudad, años, medida, cifras de la calculadora), ejecuta y redacta la respuesta final con una plantilla.

No hay listas de temas prohibidos ni regex que reescriban los argumentos del LLM.

In [1]:
import json, sys, warnings
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

warnings.filterwarnings("ignore")
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
sys.path.insert(0, str(RAIZ))
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

from agente import config, evaluacion, herramientas, motor_agente

## 2.1 · Prompt del sistema y descripciones de herramientas (v1 frente a v2)

In [2]:
print(motor_agente.prompt_sistema_agente("v2"))
for v in ("v1", "v2"):
    d = herramientas.descripciones_herramientas(v)["calcular_correlacion_o_cambio"]
    print(f"\n[{v}] calcular_correlacion_o_cambio: {d}")

Eres un agente de datos abiertos. Solo puedes responder con las herramientas: indicadores del Banco Mundial (1990-2024) y clima histórico de Open-Meteo (2020-01-01 a 2024-12-31) para los países y ciudades que listan sus descripciones. Usa exactamente los países, ciudades y años que menciona la pregunta. Si la pregunta pide algo que estas herramientas no cubren (otro tema, otro lugar u otro periodo), no llames herramientas y responde 'ABSTENCION' explicando por qué. Cuando ya tengas los datos, responde en español en 1-2 oraciones con las cifras exactas. Pasos: (1) si la pregunta nombra 2 o más países para un mismo año, usa comparar_paises_indicador; (2) si pide un dato económico de un país y el clima de una ciudad, llama las dos herramientas en el mismo turno; (3) si pide una variación porcentual, diferencia o promedio, primero consulta los datos y en el turno siguiente llama calcular_correlacion_o_cambio con los valores observados; (4) si una herramienta devuelve ERROR, corrige los arg

## 2.2 · Verificación de anclaje: reglas basadas en las capacidades de las herramientas

In [3]:
casos = [
    ("consultar_indicador_banco_mundial", {"codigo_pais": "ECU", "indicador": "pib_per_capita_usd", "anio_inicio": 2022, "anio_fin": 2022},
     "¿Cuál era la población total de Ecuador en 2022?"),
    ("consultar_clima_historico_ciudad", {"ciudad": "Lima", "fecha_inicio": "2024-01-01", "fecha_fin": "2024-12-31", "variable": "temp_media_c"},
     "¿Qué temperatura hará mañana en Lima?"),
    ("consultar_indicador_banco_mundial", {"codigo_pais": "ECU", "indicador": "pib_per_capita_usd", "anio_inicio": 2020, "anio_fin": 2023},
     "¿Cuál fue el PIB per cápita de Ecuador entre 2020 y 2023?"),
]
pd.DataFrame([{"pregunta": q, "herramienta": h, "anclado": motor_agente.verificar_anclaje(h, a, q, [])[0],
               "motivo": motor_agente.verificar_anclaje(h, a, q, [])[1]} for h, a, q in casos])

,pregunta,herramienta,anclado,motivo
0,¿Cuál era la población total de Ecuador en 2022?,consultar_indicador_banco_mundial,False,ERROR de anclaje: la pregunta no pide el indicador 'pib_per_capita_usd'.
1,¿Qué temperatura hará mañana en Lima?,consultar_clima_historico_ciudad,False,ERROR de anclaje: la pregunta no indica ningún año o periodo; no inventes fe...
2,¿Cuál fue el PIB per cápita de Ecuador entre 2020 y 2023?,consultar_indicador_banco_mundial,True,


## 2.3 · Rastro real del agente (grabado) en una pregunta de prueba con calculadora

In [4]:
version = json.loads((RAIZ / "reports" / "resultados.json").read_text(encoding="utf-8"))["version_prompt_elegida"]
pregunta = next(q["pregunta"] for q in evaluacion.cargar_conjunto("prueba") if q["id_pregunta"] == "T35")
salida = motor_agente.consultar_agente(pregunta, "tool_calling_nativo", version_prompt=version)
print("Pregunta:", pregunta)
for paso in salida["pasos"]:
    print(f"\nTurno {paso['paso']} del LLM")
    for c in paso["llamadas"]:
        print("  ", "OK " if c["exito"] else "RECHAZADA", c["herramienta"], json.dumps(c["argumentos_llm"], ensure_ascii=False))
        if not c["exito"]:
            print("     ->", c["mensaje_error"])
    if paso["texto_llm"]:
        print("   texto del LLM:", paso["texto_llm"])
print("\nRespuesta final (plantilla):", salida["respuesta_final"])

Pregunta: ¿En qué porcentaje cambió el PIB per cápita de Perú entre 2012 y 2022?

Turno 1 del LLM
   OK  consultar_indicador_banco_mundial {"codigo_pais": "PER", "indicador": "pib_per_capita_usd", "anio_inicio": 2012, "anio_fin": 2022}

Turno 2 del LLM
   OK  calcular_correlacion_o_cambio {"valores_x": [6519.38, 7353.75], "operacion": "variacion_porcentual"}

Turno 3 del LLM
   texto del LLM: El PIB per cápita de Perú aumentó en un 12.8% entre 2012 y 2022.

Respuesta final (plantilla): Según el Banco Mundial (NY.GDP.PCAP.CD), PIB per cápita (USD actuales) en Perú (PER) registró [2012: 6519.38 USD, 2013: 6746.80 USD, 2014: 6667.13 USD, 2015: 6231.71 USD, 2016: 6217.04 USD, 2017: 6736.17 USD, 2018: 6978.49 USD, 2019: 7037.01 USD, 2020: 6133.33 USD, 2021: 6828.53 USD, 2022: 7353.75 USD] (inicial 6519.38, final 7353.75, promedio 6677.21 USD). Cálculo determinista: al pasar de 6519.38 a 7353.75, el cambio absoluto es 834.37 y la variación porcentual exacta es 12.80 %.


## 2.4 · Los 4 enfoques sobre la misma pregunta de cruce (validación V21)

In [5]:
pregunta = next(q["pregunta"] for q in evaluacion.cargar_conjunto("validacion") if q["id_pregunta"] == "V21")
filas = []
for enf in ["zeroshot_sin_tools", "enrutador_reglas", "react_texto", "tool_calling_nativo"]:
    r = motor_agente.consultar_agente(pregunta, enf, version_prompt=version)
    filas.append({"enfoque": evaluacion.ETIQUETAS_ENFOQUES[enf],
                  "herramientas": ", ".join(c["herramienta"] for c in r["llamadas_herramientas"]) or "ninguna",
                  "respuesta": r["respuesta_final"][:160]})
pd.DataFrame(filas)

,enfoque,herramientas,respuesta
0,1 · LLM sin herramientas (Zero-Shot qwen2.5:7b),ninguna,El crecimiento del PIB real de Ecuador en 2023 según el Banco Mundial fue de...
1,2 · Enrutador léxico por reglas (Baseline),consultar_indicador_banco_mundial,"Según el Banco Mundial (NY.GDP.MKTP.KD.ZG), Crecimiento del PIB real anual (..."
2,3 · Agente ReAct en texto (qwen2.5:7b),ninguna,ABSTENCIÓN (ReAct): La consulta solicita información fuera del alcance de la...
3,4 · Agente tool-calling (bucle LLM + Pydantic + anclaje),"consultar_indicador_banco_mundial, consultar_clima_historico_ciudad","Según el Banco Mundial (NY.GDP.MKTP.KD.ZG), Crecimiento del PIB real anual (..."
